# Exploração inicial do dataset Wiki-RfA

## 1. Introdução

**O que é:** Uma *Request for Adminship* (RfA) é o processo onde um editor
da Wikipédia pede para virar administrador. O pedido é feito pelo próprio
candidato ou por outro membro, e qualquer editor pode votar a favor, de forma
neutra ou contra. O dataset reúne votos de RfAs desde 2003 até maio de 2013.

**Como vira um grafo:** Cada usuário é um nó. Cada voto é um arco dirigido de quem
vota (`SOURCE`) para o candidato (`TARGET`), e o voto (`VOTE`) é o peso do arco.
Como há votos contra, os pesos negativos vêm dos próprios dados.

**Onde encontrar:**
- Arquivo usado: [Wiki-RfA no Kaggle](https://www.kaggle.com/datasets/boneacrabonjac/wiki-rfa)


### O que representam SOURCE, TARGET, VOTE e RESULT?
| Coluna | O que representa |
| --- | --- |
| `SOURCE` | Usuário que vota, o eleitor. |
| `TARGET` | Usuário que recebe o voto, o candidato a administrador. |
| `VOTE` | Voto do eleitor: -1 (contra), 0 (neutro) ou 1 (a favor). |
| `RESULT` | Resultado da eleição: -1 (candidato rejeitado) ou 1 (aceito). |

Cada linha é um voto de um eleitor em um candidato numa eleição.

### O sentido dos arcos: eleitor → candidato.
Cada voto vira um arco dirigido de `SOURCE` para `TARGET` (eleitor → candidato).
O grafo é dirigido porque o voto tem direção: o fato de A votar em B não implica
que B votou em A. O peso do arco é o `VOTE`.

### O significado dos valores negativos, neutros e positivos.
| Valor | Significado |
| --- | --- |
| 1 | Voto a favor da candidatura. |
| 0 | Voto neutro. |
| -1 | Voto contra a candidatura. |

Os pesos negativos vêm dos próprios dados (votos contra).

### A ausência de data e identificador da eleição.
A versão original do dataset no SNAP traz também o ano e a data de cada voto e o
comentário do eleitor. O arquivo usado aqui tem só 4 colunas: não há data nem
identificador da eleição.

**Consequência:** cada eleição só pode ser reconhecida pelo candidato (`TARGET`).
A fonte informa que um usuário pode concorrer mais de uma vez, então um mesmo par
`SOURCE → TARGET` pode aparecer repetido por ter votado em eleições diferentes.
Sem data nem identificador, não dá para separar essas eleições nem ordenar os votos
no tempo.

### O tratamento dos dados ausentes.
- **O que encontrei:** 
- **Onde ocorrem:** 
- **Mecanismo de ausência:** 
- **Consequência para o grafo:** 
- **Tratamento escolhido:** 
- **Justificativa e limites:** 

In [3]:
import kagglehub
from pathlib import Path

# [IA] Codex – adaptar o destino do download via kaggle para a pasta do notebook
path = kagglehub.dataset_download(
    "boneacrabonjac/wiki-rfa",
    output_dir=str(Path("raw").resolve())
)

print("Path to dataset files:", path)
#pasta .complete gerada pelo proprio kagglehub indicando sucesso no download

Path to dataset files: C:\Users\andre\Documents\GRAFOS\PROJETO\projeto-grafos-g1\data\dataset_parte2\raw


In [4]:
import pandas as pd
df = pd.read_csv(
    Path(path) / "wikiRfA.csv"
, index_col=0)

df.head(10)

,SOURCE,TARGET,VOTE,RESULT
0,Steel1943,BDD,1,1
1,Cuchullain,BDD,1,1
2,INeverCry,BDD,1,1
3,Cncmaster,BDD,1,1
4,Miniapolis,BDD,1,1
5,Sven Manguard,BDD,1,1
6,Ramaksoud2000,BDD,1,1
7,RockMagnetist,BDD,1,1
8,Carrite,BDD,1,1
9,Someguy1221,BDD,1,1


In [5]:
df.shape

(198275, 4)

In [6]:
df.dtypes

SOURCE      str
TARGET      str
VOTE      int64
RESULT    int64
dtype: object

In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 198275 entries, 0 to 198274
Data columns (total 4 columns):
 #   Column  Non-Null Count   Dtype
---  ------  --------------   -----
 0   SOURCE  196614 non-null  str  
 1   TARGET  198275 non-null  str  
 2   VOTE    198275 non-null  int64
 3   RESULT  198275 non-null  int64
dtypes: int64(2), str(2)
memory usage: 6.1 MB


In [8]:
resultado = df["TARGET"].nunique()
contagem =df["TARGET"].value_counts()
print("Número de classes únicas na coluna TARGET:", resultado)
print("Contagem de cada classe na coluna TARGET:")
print(contagem)
# mostra os nós que receberam mais e menos votos em target

Número de classes únicas na coluna TARGET: 3497
Contagem de cada classe na coluna TARGET:
TARGET
Everyking         987
TenPoundHammer    778
MZMcBride         663
Cobi              639
Elonka            628
                 ... 
Pcb21               1
Schneelocke         1
Stevertigo          1
Tompagenet          1
Zanimum             1
Name: count, Length: 3497, dtype: int64


In [9]:
df.isna().sum()

SOURCE    1661
TARGET       0
VOTE         0
RESULT       0
dtype: int64

In [10]:
df[df['SOURCE'].isna()]

,SOURCE,TARGET,VOTE,RESULT
707,NaN,Jason Quinn,0,1
708,NaN,Jason Quinn,0,1
793,NaN,Legoktm,1,1
1126,NaN,Mattythewhite,-1,1
1128,NaN,Mattythewhite,-1,1
...,...,...,...,...
198052,NaN,Yelyos,1,-1
198116,NaN,Christopher Mahan,1,1
198171,NaN,Jwrosenzweig,1,1
198217,NaN,Patrick,1,1


In [11]:
# [IA] ChatGPT (Codex) – apoio com sintaxe de pandas para conferir qualidade, sem alterar os dados
# Executar após a leitura do CSV e antes da célula que aplica dropna.
colunas = ["SOURCE", "TARGET", "VOTE", "RESULT"]
extremos_presentes = df["SOURCE"].notna() & df["TARGET"].notna()
pares = df.loc[extremos_presentes, ["SOURCE", "TARGET"]]

print("Registros antes do tratamento:", len(df))
print("Valores ausentes por coluna:")
print(df[colunas].isna().sum())
print("Linhas idênticas nas quatro colunas, além da primeira ocorrência:",
      df.duplicated(subset=colunas).sum())
print("Laços com SOURCE e TARGET presentes:",
      (df.loc[extremos_presentes, "SOURCE"] == df.loc[extremos_presentes, "TARGET"]).sum())
print("Registros com par SOURCE/TARGET repetido, incluindo todas as ocorrências:",
      pares.duplicated(keep=False).sum())
print("Ocorrências de pares além da primeira:", pares.duplicated().sum())
print("VOTE preenchido fora de {-1, 0, 1}:",
      (df["VOTE"].notna() & ~df["VOTE"].isin([-1, 0, 1])).sum())
print("RESULT preenchido fora de {-1, 1}:",
      (df["RESULT"].notna() & ~df["RESULT"].isin([-1, 1])).sum())


Registros antes do tratamento: 198275
Valores ausentes por coluna:
SOURCE    1661
TARGET       0
VOTE         0
RESULT       0
dtype: int64
Linhas idênticas nas quatro colunas, além da primeira ocorrência: 3399
Laços com SOURCE e TARGET presentes: 103
Registros com par SOURCE/TARGET repetido, incluindo todas as ocorrências: 16330
Ocorrências de pares além da primeira: 8537
VOTE preenchido fora de {-1, 0, 1}: 0
RESULT preenchido fora de {-1, 1}: 0


### Conferência de votos sem SOURCE por candidato

Contagens no arquivo original e verificação de presença como eleitor após a exclusão de registros sem SOURCE ou TARGET.


In [ ]:
# [IA] ChatGPT (Codex) – apoio com pandas para conferir faltantes por candidato
# Relê o arquivo original para incluir os registros anteriores ao dropna.
dados_conferencia = pd.read_csv(Path(path) / "wikiRfA.csv", index_col=0)
faltantes_source = dados_conferencia["SOURCE"].isna()
votos_totais = dados_conferencia.groupby("TARGET").size().rename("votos_totais")
votos_sem_source = dados_conferencia.loc[faltantes_source].groupby("TARGET").size().rename("sem_SOURCE")
conferencia_candidatos = pd.concat([votos_totais, votos_sem_source], axis=1)
conferencia_candidatos["sem_SOURCE"] = conferencia_candidatos["sem_SOURCE"].fillna(0).astype(int)
conferencia_candidatos["percentual_sem_SOURCE"] = 100 * conferencia_candidatos["sem_SOURCE"] / conferencia_candidatos["votos_totais"]
print("Registros originais:", len(dados_conferencia))
print("Registros sem SOURCE:", int(faltantes_source.sum()))
print("Candidatos distintos:", len(conferencia_candidatos))
print("Candidatos com votos sem SOURCE:", int((conferencia_candidatos["sem_SOURCE"] > 0).sum()))
print("\nMaiores contagens de votos sem SOURCE:")
print(conferencia_candidatos.sort_index().sort_values("sem_SOURCE", ascending=False, kind="stable").head(20).to_string())
print("\nMaiores percentuais entre os candidatos afetados:")
print(conferencia_candidatos.loc[conferencia_candidatos["sem_SOURCE"] > 0].sort_index().sort_values("percentual_sem_SOURCE", ascending=False, kind="stable").head(20).to_string())


Registros originais: 198275
Registros sem SOURCE: 1661
Candidatos distintos: 3497
Candidatos com votos sem SOURCE: 926

Maiores contagens de votos sem SOURCE:
                     votos_totais  sem_SOURCE  percentual_sem_SOURCE
TARGET                                                              
Tawker                        229          28              12.227074
FT2                           139          13               9.352518
MatthewFenton                 116          13              11.206897
No Guru                        80          13              16.250000
Andypandy.UK                  107          12              11.214953
ILovePlankton                 109          12              11.009174
Lightdarkness                 149          12               8.053691
Lord Voldemort                245          12               4.897959
Misza13                       143          12               8.391608
RGTraynor                      50          12              24.000000
TodorBozhinov

In [ ]:
# [IA] ChatGPT (Codex) – apoio com pandas para verificar os candidatos com 100% dos votos recebidos sem SOURCE
candidatos_sem_origem = conferencia_candidatos.loc[
    conferencia_candidatos["sem_SOURCE"] == conferencia_candidatos["votos_totais"]
].copy()
registros_mantidos = dados_conferencia.dropna(subset=["SOURCE", "TARGET"])
contagem_eleitores = registros_mantidos["SOURCE"].value_counts()
candidatos_sem_origem["votos_como_eleitor_mantidos"] = contagem_eleitores.reindex(candidatos_sem_origem.index, fill_value=0)
print(candidatos_sem_origem.sort_index().to_string())


              votos_totais  sem_SOURCE  percentual_sem_SOURCE  votos_como_eleitor_mantidos
TARGET                                                                                    
Lionelisbest             1           1                  100.0                            0
Meelar                  11          11                  100.0                           92
Mrmewe                   1           1                  100.0                            1
Patrick                  1           1                  100.0                            0
Sarge Baldy              8           8                  100.0                           52
Stevertigo               1           1                  100.0                            0


In [12]:
#remove votos sem identificação de quem votou
antes = len(df)

df = df.dropna(subset=["SOURCE", "TARGET"]).copy()

print("Antes:", antes)
print("Depois:", len(df))
print("Removidos:", antes - len(df))

Antes: 198275
Depois: 196614
Removidos: 1661


In [13]:
print(df["VOTE"].value_counts().sort_index())
print(df["RESULT"].value_counts().sort_index())

VOTE
-1     40667
 0     12068
 1    143879
Name: count, dtype: int64
RESULT
-1     74090
 1    122524
Name: count, dtype: int64


In [14]:
resultados = df.groupby("TARGET")["RESULT"].nunique()

print("Candidatos com resultados diferentes:", (resultados > 1).sum())

Candidatos com resultados diferentes: 236
